Onsets are not matched with MRI data!!!! Really only a script ot extract reaction times and accuracy

In [2]:
import nilearn
import numpy as np
import pandas as pd
import os
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as op
import hcp_utils as hcp
from statsmodels.stats.multitest import multipletests
import glob

from brainspace.utils.parcellation import map_to_labels
from infomap import Infomap
#from my_utils import get_glasser_parcels, get_glasser_CAatlas_mapping, get_basic_mask
from matplotlib.colors import ListedColormap
import warnings

warnings.filterwarnings("ignore")

pixdim[1,2,3] should be non-zero; setting 0 dims to 1


In [24]:
# getting the raw files

task = 'magjudg'
file_task = 'risk'
sessions = ['2']#, '2', '3'] # is hard-coded, sorry didnt know where else to get this information from
runs = ['1', '2', '3'] # is hard-coded, sorry didnt know where else to get this information from

main_path = '/mnt_AdaBD_largefiles/Data/SMILE_Data/measurements'
bids_folder = '/mnt_AdaBD_largefiles/Data/DNumrisk_Data/ds-smile'

path_logfiles = op.join(main_path, f'{task}_logfiles')

subList = sorted([f[4:7] for f in os.listdir(bids_folder)
                      if f.startswith("sub-") and len(f) == 7], key=lambda x: int(x))

subList = ['106']

for sub in subList:
    for ses in sessions:
        dfs = []
        for run in runs:
            try:
                path = op.join(path_logfiles, f'sub-{sub}', f'ses-{ses}', f'sub-{sub}_ses-{ses}_task-{file_task}_run-{run}_events.tsv')
                behavior = pd.read_csv(path, delimiter='\t')

                behavior['trial_nr'] = behavior['trial_nr'].astype(int)
                behavior = behavior[behavior['trial_nr'] > 0]

                stim1 = behavior[(behavior['event_type'] == 'stim') & (behavior['phase'] == 2)].copy()
                stim1['n'] = stim1['n1']
                stim1['trial_type'] = 'stimulus 1'


                stim2 = behavior[(behavior['event_type'] == 'stim') & (behavior['phase'] == 4)].copy()
                stim2['n'] = stim2['n2']
                stim2['trial_type'] = 'stimulus 2'


                choice = behavior[(behavior['event_type'] == 'choice')].copy()
                choice['trial_type'] = 'choice'

                events = pd.concat((stim1, stim2, choice)).sort_index().reset_index(drop=True)
                events = events[['trial_nr', 'onset', 'trial_type', 'n1', 'n2', 'choice']]
                events['reaction_time'] = np.nan
                events['accuracy'] = np.nan

                for trial in events['trial_nr'].unique():
                    try:
                        onset_stim2 = events[events['trial_nr'] == trial][events['trial_type'] == 'stimulus 2']['onset'].values[0]
                        onset_choice = events[events['trial_nr'] == trial][events['trial_type'] == 'choice']['onset'].values[0]

                        index = events[events['trial_nr'] == trial][events['trial_type'] == 'choice'].index[0]
                        events['reaction_time'][index] = onset_choice - onset_stim2

                        n1 = events[events['trial_nr'] == trial]['n1'].unique()[0].astype(int)
                        n2 = events[events['trial_nr'] == trial]['n2'].unique()[0].astype(int)
                        choice = events[events['trial_nr'] == trial][events['trial_type'] == 'choice']['choice'].values[0].astype(int)
                        events['accuracy'][index] = (1 if (n1 > n2 and choice == 1) or (n1 < n2 and choice == 2)
                                                    else 0 if (n1 > n2 and choice == 2) or (n1 < n2 and choice == 1)
                                                    else np.nan)
                    except (IndexError, KeyError):
                        continue

                events['run'] = run
                events['ses'] = ses

                dfs.append(events)
            except:
                print(f'sub-{sub} ses-{ses} run-{run} not found, skipping')
                continue
        if dfs:
            events_all = pd.concat(dfs, ignore_index=True)
            events_all.to_csv(op.join(bids_folder, f'derivatives/behavioral_files/magjudge_files/sub-{sub}_ses-{ses}_magjudge-behav.csv'), index=False)
        else:
            print(f'sub-{sub} ses-{ses} has no runs, skipping')

In [25]:
behavior

,trial_nr,onset,event_type,phase,response,nr_frames,n1,n2,choice,onset_abs,duration
0,3,0.007153,stim,0,NaN,1.0,NaN,NaN,NaN,1.992852,0.021686
1,3,-1.082905,response,0,d,NaN,NaN,NaN,NaN,0.902794,NaN
973,61,16.719098,stim,0,NaN,15.0,20.0,23.0,NaN,18.704797,0.250164
974,61,16.766863,response,0,lshift,NaN,20.0,23.0,NaN,18.752562,NaN
975,61,16.766966,pulse,0,t,NaN,20.0,23.0,NaN,18.752666,NaN
...,...,...,...,...,...,...,...,...,...,...,...
24830,91,444.332186,pulse,0,t,NaN,NaN,NaN,NaN,446.317885,NaN
24831,91,444.347704,response,0,lshift,NaN,NaN,NaN,NaN,446.333403,NaN
24832,91,444.347775,pulse,0,t,NaN,NaN,NaN,NaN,446.333474,NaN
24833,91,444.397851,response,0,lshift,NaN,NaN,NaN,NaN,446.383550,NaN


In [ ]:
# same code for dnumrisk magjudge files
# getting the raw files

task = 'magjudge'
sessions = ['1'] # is hard-coded, sorry didnt know where else to get this information from
runs = ['1', '2', '3', '4', '5', '6'] # is hard-coded, sorry didnt know where else to get this information from

main_path = '/mnt_03/ds-dnumrisk/'
bids_folder = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk'

path_logfiles = main_path

subList = sorted([f[4:7] for f in os.listdir(op.join(bids_folder, 'derivatives/glm_stim2.denoise'))
                      if f.startswith("sub-") and len(f) == 6], key=lambda x: int(x))

#subList = ['101']

for sub in subList:
    for ses in sessions:
        dfs = []
        for run in runs:
            try:
                path = op.join(path_logfiles, f'sub-{sub}', f'ses-{ses}', 'func', f'sub-{sub}_ses-{ses}_task-{task}_run-{run}_events.tsv')
                events = pd.read_csv(path, delimiter='\t')

                events['trial_nr'] = events['trial_nr'].astype(int)
                events = events[events['trial_nr'] > 0]

                events['reaction_time'] = np.nan
                events['accuracy'] = np.nan

                for trial in events['trial_nr'].unique():
                    try:
                        onset_stim2 = events[events['trial_nr'] == trial][events['trial_type'] == 'stimulus 2']['onset'].values[0]
                        onset_choice = events[events['trial_nr'] == trial][events['trial_type'] == 'choice']['onset'].values[0]

                        index = events[events['trial_nr'] == trial][events['trial_type'] == 'choice'].index[0]
                        events['reaction_time'][index] = onset_choice - onset_stim2

                        n1 = events[events['trial_nr'] == trial]['n1'].unique()[0].astype(int)
                        n2 = events[events['trial_nr'] == trial]['n2'].unique()[0].astype(int)
                        choice = events[events['trial_nr'] == trial][events['trial_type'] == 'choice']['choice'].values[0].astype(int)
                        events['accuracy'][index] = (1 if (n1 > n2 and choice == 1) or (n1 < n2 and choice == 2)
                                                    else 0 if (n1 > n2 and choice == 2) or (n1 < n2 and choice == 1)
                                                    else np.nan)
                    except (IndexError, KeyError):
                        continue

                events['run'] = run
                events['ses'] = ses

                dfs.append(events)
            except:
                print(f'sub-{sub} ses-{ses} run-{run} not found, skipping')
                continue
        if dfs:
            events_all = pd.concat(dfs, ignore_index=True)
            events_all.to_csv(op.join(bids_folder, f'derivatives/behavioral_files/sub-{sub}_ses-{ses}_magjudge-behav.csv'), index=False)
        else:
            print(f'sub-{sub} ses-{ses} has no runs, skipping')